# 04 · Regressions

$$R^{overnight}_{i,t+1} = \alpha_i + \beta_1 R^{close}_{i,t} + \beta_2 AVOL_{i,t} + \beta_3 R^{close}_{i,t}AVOL_{i,t} + \gamma X_{i,t} + \epsilon$$

Ticker fixed effects, two-way clustering by ticker and date. Returns are in
basis points.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from closingbell import config as C, calendar_utils as cal

def table(name):
    return pd.read_csv(C.TABLES / f"{name}.csv")

sessions = pd.read_parquet(C.PROCESSED / "sessions.parquet")
print(f"{len(sessions):,} ticker-sessions, {sessions.session.min()} to {sessions.session.max()}")

15,756 ticker-sessions, 2021-01-04 to 2026-03-31


In [2]:
from closingbell import events as ev, regressions as rx
universe = ev.event_panel(sessions)
main = rx.fit_ols(universe, cluster="two_way")
rx.tidy(main).round(4)

,term,coef,se,t,p,ci_low,ci_high
0,const,1.8452,9.8872,0.1866,0.8520,-17.5333,21.2236
1,r_close,-0.1437,0.0645,-2.2279,0.0259,-0.2701,-0.0173
2,avol,9.6026,5.9944,1.6019,0.1092,-2.1462,21.3514
3,r_close_x_avol,-0.0061,0.0958,-0.0639,0.9490,-0.1939,0.1816
4,mkt_r_session,0.0264,0.0485,0.5449,0.5858,-0.0686,0.1214
5,realized_vol,-223.5739,854.0383,-0.2618,0.7935,-1897.4582,1450.3103
6,r_overnight_prev,-0.0015,0.0199,-0.0747,0.9405,-0.0405,0.0375
7,month_end,5.9102,10.5407,0.5607,0.5750,-14.7492,26.5696
8,dow_Tuesday,2.6329,6.7669,0.3891,0.6972,-10.6299,15.8957
9,dow_Wednesday,0.5669,8.4699,0.0669,0.9466,-16.0338,17.1676


`r_close` is the coefficient of interest: about −0.14, so 100 bp of closing move
is followed by roughly 14 bp of overnight give-back. The interaction is
essentially a precise zero — abnormal volume does not modulate the relationship.

In [3]:
print(f"n = {int(main.nobs):,}   R^2 = {main.rsquared:.4f}")

n = 14,945   R^2 = 0.0048


## Does the conclusion depend on the error structure?

Date clustering is the one that matters: twelve correlated names share every trading day.

In [4]:
table('regression_se_comparison').round(4)

,cov,term,coef,se,t,p
0,two_way,r_close,-0.1437,0.0645,-2.2279,0.0259
1,two_way,avol,9.6026,5.9944,1.6019,0.1092
2,two_way,r_close_x_avol,-0.0061,0.0958,-0.0639,0.9490
3,date,r_close,-0.1437,0.0721,-1.9916,0.0464
4,date,avol,9.6026,5.4407,1.7650,0.0776
5,date,r_close_x_avol,-0.0061,0.1021,-0.0600,0.9522
6,ticker,r_close,-0.1437,0.0239,-6.0013,0.0000
7,ticker,avol,9.6026,4.4026,2.1811,0.0292
8,ticker,r_close_x_avol,-0.0061,0.0608,-0.1008,0.9197
9,hc1,r_close,-0.1437,0.0402,-3.5712,0.0004


Ticker-only clustering gives a t near −6 because it ignores exactly the dependence that dominates here. It is reported for completeness, never as the headline.

## Specification ladder

In [5]:
sl = table('regression_spec_ladder')
sl[sl.term == "r_close"][["spec", "coef", "se", "t", "p", "n", "r2"]].round(4)

,spec,coef,se,t,p,n,r2
0,1: closing return only,-0.1323,0.0666,-1.9881,0.0468,14958,0.0020
3,2: + ticker FE,-0.1337,0.0668,-2.0026,0.0452,14958,0.0031
6,3: + market & vol controls,-0.1466,0.0653,-2.2448,0.0248,14958,0.0035
9,4: + prev overnight,-0.1462,0.0652,-2.2435,0.0249,14945,0.0035
12,5: full (calendar controls),-0.1437,0.0645,-2.2279,0.0259,14945,0.0048


## Is it mechanical?

`r_close30` ends at the closing price and `r_overnight` begins there, so measurement error in that one price enters the two returns with opposite signs. Re-measuring the closing move to the last regular trade removes the shared price entirely.

In [6]:
for name, lab in [("regression_main", "shares the closing price"),
                  ("regression_no_shared_price", "no shared price"),
                  ("regression_close60", "60-minute window"),
                  ("regression_ex_corp_actions", "ex split/dividend sessions"),
                  ("regression_events_only", "extreme events only")]:
    r = table(name).set_index("term").loc["r_close"]
    print(f"{lab:32s} coef {r.coef:+.4f}  t {r.t:+.2f}  p {r.p:.3f}")

shares the closing price         coef -0.1437  t -2.23  p 0.026
no shared price                  coef -0.1384  t -2.13  p 0.033
60-minute window                 coef -0.1541  t -2.76  p 0.006
ex split/dividend sessions       coef -0.1389  t -2.17  p 0.030
extreme events only              coef -0.0913  t -0.93  p 0.352


The coefficient survives removing the shared price, so the negative sign is not
a microstructure artefact. It does **not** survive restricting to extreme events
alone, which reinforces the decile picture: this is a broad tilt, not a tail
phenomenon.

## Per-ticker

In [7]:
table('per_ticker_coefficients').round(3)

,ticker,coef,se,t,p,ci_low,ci_high,n
0,AMZN,-0.299,0.138,-2.173,0.030,-0.568,-0.029,1242
1,NVDA,-0.226,0.103,-2.192,0.028,-0.429,-0.024,1250
2,MSFT,-0.165,0.103,-1.608,0.108,-0.366,0.036,1250
3,IWM,-0.133,0.090,-1.476,0.140,-0.309,0.043,1248
4,QQQ,-0.106,0.105,-1.009,0.313,-0.313,0.100,1251
5,TSLA,-0.097,0.099,-0.980,0.327,-0.291,0.097,1244
6,GOOGL,-0.088,0.106,-0.826,0.409,-0.296,0.121,1231
7,META,-0.086,0.137,-0.625,0.532,-0.355,0.184,1249
8,AAPL,-0.072,0.117,-0.615,0.538,-0.301,0.157,1243
9,JPM,-0.070,0.103,-0.675,0.499,-0.271,0.132,1252


All twelve are negative; two are individually significant. The pooled estimate is not one name's result.

See `results/figures/fig10`.

## Persistence classifier

In [8]:
extreme = universe[universe.is_extreme]
logit = rx.fit_persistence_logit(extreme)
rx.tidy(logit).round(4)

,term,coef,se,t,p,ci_low,ci_high
0,const,0.2626,0.3292,0.7978,0.4250,-0.3826,0.9079
1,abs_pressure,-0.0177,0.1367,-0.1295,0.8969,-0.2857,0.2503
2,direction,0.2988,0.0876,3.4095,0.0007,0.1270,0.4706
3,avol,0.3014,0.3646,0.8266,0.4085,-0.4132,1.0159
4,abs_pressure_x_avol,-0.1396,0.1496,-0.9333,0.3506,-0.4328,0.1536
5,realized_vol,0.0002,0.0012,0.1638,0.8699,-0.0022,0.0026
6,mkt_r_session,-0.0003,0.0009,-0.3432,0.7315,-0.0021,0.0015
7,month_end,-0.3383,0.2444,-1.3841,0.1663,-0.8173,0.1407


In [9]:
import json
fit = rx.brier_and_auc(logit)
print(json.dumps(fit, indent=2))
print()
print("Brier improvement over the base rate: %.4f" % (fit["brier_baseline"] - fit["brier"]))

{
  "n": 1974,
  "base_rate": 0.4635258358662614,
  "brier": 0.24108217194209924,
  "brier_baseline": 0.24866963535074507,
  "auc": 0.5997688302708505
}

Brier improvement over the base rate: 0.0076


Only `direction` is significant, and that coefficient is the overnight drift restated. AUC near 0.60 in-sample, with a Brier score barely better than always predicting the base rate.

In [10]:
table('persistence_calibration').round(4)

,bin,n,mean_predicted,observed,obs_ci_low,obs_ci_high
0,1,198,0.3245,0.2576,0.1967,0.3185
1,2,197,0.3679,0.3655,0.2982,0.4327
2,3,197,0.3887,0.4162,0.3474,0.4851
3,4,198,0.4129,0.4293,0.3603,0.4982
4,5,197,0.4443,0.4467,0.3773,0.5161
5,6,197,0.4845,0.5431,0.4736,0.6127
6,7,198,0.5103,0.5253,0.4557,0.5948
7,8,197,0.5332,0.5482,0.4787,0.6177
8,9,197,0.5617,0.5330,0.4633,0.6027
9,10,198,0.6071,0.5707,0.5018,0.6397


Calibration is reasonable, but predicted probabilities span only about 0.32 to 0.61 — the model has very little to say.